# Phase 3: LLM Integration
This notebook demonstrates how to connect our existing retrieval system to a Large Language Model (LLM).

The LLM backend is controlled by the `LLM_BACKEND` variable in `.env`:
- Set `LLM_BACKEND=ollama` to use a local Ollama model (default)
- Set `LLM_BACKEND=openai` to use OpenAI's API

In [1]:
import os
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv(dotenv_path="../.env")

True

## System Initialization

This cell initializes the two core components of our RAG system:
1. **Vector Store**: Connects to the local ChromaDB database containing your document chunks.
2. **LLM Backend**: Loads the Large Language Model. The system automatically switches between **OpenAI** (cloud) and **Ollama** (local) based on the `LLM_BACKEND` setting in your `.env` file.

> **Note**: If using OpenAI, ensure your API key is valid. If using Ollama, ensure the Ollama service is running locally.

In [ ]:
# Model & Storage Settings
EMBEDDING_MODEL_NAME = os.getenv("EMBEDDING_MODEL_NAME")
COLLECTION_NAME = os.getenv("COLLECTION_NAME")
CHROMA_DB_PATH = os.getenv("CHROMA_DB_PATH")
LLM_BACKEND = os.getenv("LLM_BACKEND", "ollama").lower()

# Environment Validation
if not EMBEDDING_MODEL_NAME:
    raise ValueError("EMBEDDING_MODEL_NAME is not set in the environment or .env file.")
if not COLLECTION_NAME:
    raise ValueError("COLLECTION_NAME is not set in the environment or .env file.")
if not CHROMA_DB_PATH:
    raise ValueError("CHROMA_DB_PATH is not set in the environment or .env file.")

# 1. Initialize Vector Store
embedding_function = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_function,
    persist_directory=CHROMA_DB_PATH
)

# 2. Initialize LLM based on LLM_BACKEND flag from .env
if LLM_BACKEND == "openai":
    from langchain_openai import ChatOpenAI

    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    if not OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY is not set in the environment or .env file.")
        
    llm = ChatOpenAI(temperature=0, api_key=OPENAI_API_KEY)
    print(f"System loaded! Using OpenAI backend.")
else:
    from langchain_community.llms import Ollama

    OLLAMA_MODEL = os.getenv("OLLAMA_MODEL")
    OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
    if not OLLAMA_MODEL:
        raise ValueError("OLLAMA_MODEL is not set in the environment or .env file.")

    llm = Ollama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL)
    print(f"System loaded! Using Ollama backend (model: {OLLAMA_MODEL}).")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


System loaded! Using OpenAI backend.


### Step 1: Query the vector store

In [3]:
query = "What is the main topic of the document?"

# Initialize retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Retrieve context pieces
docs = retriever.invoke(query)
context_pieces = [doc.page_content for doc in docs]

context_str = "\n\n".join(context_pieces)
print("Retrieved Context:\n", context_str[:200], "...")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retrieved Context:
 CityRAG 23
Fig. 11: User study Q3.

is given a context and asked to determine which entity a pronoun refers to, requiring the model to
exhibit commonsense knowledge and contextual understanding. Winog ...


### Step 2: Generate Answer

In [4]:
template = """You are a helpful AI assistant. Use the following pieces of retrieved context to answer the user's question. 
If you don't know the answer based on the context, just say that you don't know. Do not make up information.

--- CONTEXT ---
{context}
---------------

User Question: {query}
Answer:"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "query"]
)

# Create the chain and execute it
chain = prompt | llm
answer = chain.invoke({
    "context": context_str,
    "query": query
})

print("Answer:\n", answer)

Answer:
 content='The main topic of the document appears to be about different question answering datasets such as Winogrande, OpenBookQA, and BoolQ, which require models to exhibit commonsense knowledge and contextual understanding.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 312, 'total_tokens': 353, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DXQooyWMdI4daxuwh0Wu3qGob6QoJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019db520-461e-7f83-9c66-1256afac86dd-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 312, 'output_tokens': 41, 'total_tokens': 353, 'input_token_details': {'audio': 0, 'cache_